# Deuteron NMR Lineshape Fit — Example
Fits an example signal from Run Group C at Jefferson Lab using `deuteron_fit.fit`.

In [ ]:
from deuteron_fit import fit
import matplotlib.pyplot as plt
import json

## Load example data

In [ ]:
with open('example_data/example_data.json', 'r') as event:
    for line in event:
        json_dict = json.loads(line.rstrip('\n|\r'))
        signal = json_dict['fitsub']
        freqs  = json_dict['freq_list']

print(f'Loaded {len(freqs)} frequency points')

## Set initial parameters and run fit

In [ ]:
initial_params = {
    'A':   0.03,
    'G':  -0.00003,
    'r':   1.2,      # r > 1 is positive polarization
    'wQ':  0.027,
    'wL':  32.69,
    'eta': -0.02,
    'xi':  -0.001,
}

result = fit(freqs, signal, initial_params)
print(result.fit_report())

## Calculate polarization

The asymmetry parameter `r` gives the vector polarization P and tensor polarization P$_{zz}$:

$$P = \frac{r^2 - 1}{r^2 + r + 1} \qquad P_{zz} = \frac{(r-1)^2}{r^2 + r + 1}$$

Errors are propagated from the fit uncertainty on `r`.

In [ ]:
r      = result.params['r'].value
r_err  = result.params['r'].stderr
denom  = r**2 + r + 1

P      = (r**2 - 1) / denom
Pzz    = (r - 1)**2 / denom

P_err   = (r**2 + 4*r + 1) / denom**2 * r_err
Pzz_err = 3 * abs(r**2 - 1)  / denom**2 * r_err

print(f'Vector polarization:  P   = {P*100:.2f} ± {P_err*100:.2f}%')
print(f'Tensor polarization:  Pzz = {Pzz*100:.2f} ± {Pzz_err*100:.2f}%')

## Plot signal and fit

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(freqs, signal, label='Signal')
ax.plot(freqs, result.best_fit, '-r', label='Fit')
ax.set_xlabel('Frequency')
ax.set_ylabel('Signal')
ax.set_title(f'NMR Signal  |  P = {P*100:.2f}%  Pzz = {Pzz*100:.2f}%')
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()